In [10]:
import os
#os.chdir("../")
%pwd

'/home/abood/datascienceproject'

In [11]:
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/aboodcs/datascienceproject.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "aboodcs"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "6101bda6b23aa9ef440664cb2f0aac4cb03e1bcf"

In [12]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metrics_file_path: Path
    all_params: dict
    mlflow_uri: str
    target_column: str

In [13]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories,save_json

In [14]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir = config.root_dir,
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            metrics_file_path = config.metrics_file_path,
            all_params = params,
            mlflow_uri = "https://dagshub.com/aboodcs/datascienceproject.mlflow",
            target_column = schema.name
        )
        return model_evaluation_config

In [15]:
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import mlflow
import mlflow.sklearn
import numpy as np
from urllib.parse import urlparse
from tqdm.notebook import tqdm as notebook_tqdm

In [16]:
class ModelEvaluation:
    def __init__(self,config: ModelEvaluationConfig):
        self.config = config
    def eval_metrics(self,actual , pred):
        rmse = np.sqrt(mean_squared_error(actual , pred))
        mae = mean_absolute_error(actual , pred)
        r2 = r2_score(actual , pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[self.config.target_column]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)
            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
            mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

            #saving metrices as local
            scores = {
                "rmse": rmse,
                "mae": mae,
                "r2": r2}
            save_json(path=Path(self.config.metrics_file_path), data=scores)

            mlflow.log_params(self.config.all_params)
            
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticNetCPUModel")
            else:
                mlflow.sklearn.log_model(model, "model")

In [17]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-05-06 13:27:06,468: INFO: common]: yaml file: config/config.yaml loaded successfully
[2026-05-06 13:27:06,470: INFO: common]: yaml file: params.yaml loaded successfully
[2026-05-06 13:27:06,472: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-05-06 13:27:06,473: INFO: common]: created directory at: artifacts
[2026-05-06 13:27:06,473: INFO: common]: created directory at: artifacts/model_evaluation
[2026-05-06 13:27:07,268: INFO: common]: json file saved at: artifacts/model_evaluation/metrics.yaml


2026/05/06 13:27:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 13:27:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'ElasticNetCPUModel' already exists. Creating a new version of this model...
2026/05/06 13:27:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNetCPUModel, version 6
Created version '6' of model 'ElasticNetCPUModel'.


🏃 View run efficient-kite-607 at: https://dagshub.com/aboodcs/datascienceproject.mlflow/#/experiments/0/runs/6cdb77f6b8e1450b9d79351928c6f9df
🧪 View experiment at: https://dagshub.com/aboodcs/datascienceproject.mlflow/#/experiments/0
